# LightningPoseTrack on Colab — Setup + Auto-Runner

This notebook does two things:
1. Installs everything (packages, opencode CLI, Drive mount, repo clone)
2. Gives you a button to run any notebook and auto-catch errors

**Runtime**: Python 3, **GPU** (T4/V100/A100)

---
## The Fix Loop

```
┌─────────────────────┐     ┌──────────────────┐     ┌─────────────┐
│  Run notebook       │────>│  Error?           │────>│  Tell        │
│  on Colab           │     │  Saved to file    │     │  opencode    │
└─────────────────────┘     └──────────────────┘     └──────┬──────┘
       ▲                                                     │
       │                     ┌──────────────────┐            │
       │                     │  opencode fixes   │<───────────┘
       │                     │  → pushes to GH   │
       │                     └──────────────────┘
       │                             │
       └─────────────────────────────┘
       git pull & re-run
```

In [ ]:
# ===== CONFIGURATION =====
GITHUB_REPO = "https://github.com/kaarthik-balakrishnan/LightningPoseTrack.git"
GIT_BRANCH = "main"
REPO_DIR = "/content/LightningPoseTrack"

# ============================================================
# DRIVE PATHS — set each one to match your Google Drive layout
# ============================================================
# Raw video session folders (often on a Shared drive)
DRIVE_RAW_VIDEOS = "/content/drive/Shareddrives/3R.Data/.../260529.00000002/"  # ← change

# Sampled frames for labeling (output from 02, input to 03)
DRIVE_LABELED = "/content/drive/My Drive/PigBehavior/labeled_frames"  # ← change

# Computed background images (output from 02)
DRIVE_BACKGROUNDS = "/content/drive/My Drive/PigBehavior/backgrounds"  # ← change

# Trained model checkpoints (output from 03/07, input to 04)
DRIVE_MODELS = "/content/drive/My Drive/PigBehavior/trained_models"  # ← change

# Pose prediction CSVs (output from 04, input to 05/06/08)
DRIVE_POSE_OUTPUTS = "/content/drive/My Drive/PigBehavior/pose_outputs"  # ← change

# Feature CSVs (output from 05, input to 07/08)
DRIVE_FEATURES = "/content/drive/My Drive/PigBehavior/features"  # ← change

# Behavior detection CSVs (output from 06, input to 08)
DRIVE_BEHAVIOR_OUTPUTS = "/content/drive/My Drive/PigBehavior/behavior_outputs"  # ← change

# Daily report files (output from 08)
DRIVE_REPORTS = "/content/drive/My Drive/PigBehavior/reports"  # ← change

# ---- do not change below this line ----
# Names of all Drive path variables (so colab_runner can inject them)
DRIVE_PATH_NAMES = [
    "DRIVE_RAW_VIDEOS", "DRIVE_LABELED", "DRIVE_BACKGROUNDS",
    "DRIVE_MODELS", "DRIVE_POSE_OUTPUTS", "DRIVE_FEATURES",
    "DRIVE_BEHAVIOR_OUTPUTS", "DRIVE_REPORTS",
]

## Step 1: Mount Drive & Clone Repo

In [ ]:
import os, sys
from google.colab import drive
drive.mount("/content/drive")

if not os.path.exists(REPO_DIR):
    !git clone -b {GIT_BRANCH} {GITHUB_REPO} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/src")
print(f"Repo at {REPO_DIR}")

## Step 2: Install Everything

In [ ]:
print("=== System packages ===")
!apt-get update -qq && apt-get install -y -qq ffmpeg libgl1 libglib2.0-0 tesseract-ocr

print("=== Python dependencies ===")
!pip install -q -r requirements-colab.txt
!pip install -q lightning-pose[all] omegaconf
!pip install -q scikit-learn xgboost joblib

print("=== opencode CLI ===")
!curl -fsSL https://opencode.ai/install | bash

print("\\n=== Verify ===")
import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
!opencode --version 2>/dev/null || echo "opencode installed"
print("\\nAll done. Ready to run notebooks.")

---
## Step 3: Run a Notebook

Use the runner script below. It executes the notebook, captures errors,
and saves the traceback so you can share it with opencode.

**How to use:**
1. Run the cell below with the notebook number you want
2. If it succeeds → move to the next notebook
3. If it fails → error is saved to `/tmp/colab_error.txt`
4. Paste that error to opencode (in your local terminal or where opencode runs)
5. opencode fixes the code and pushes to GitHub
6. Run the cell again — it auto-pulls latest code and re-runs

**Sequence (recommended):** 01 → 02 → 03 → 04 → 05 → 06 → 07 → 08

In [ ]:
# Pick which notebook to run. Change the number below.
NOTEBOOK_NUM = "01"  # ← change this: "01" through "08"

import subprocess, sys
from pathlib import Path

# Map number → filename
NOTEBOOKS = {
    "01": "01_Data_Audit",
    "02": "02_Frame_Sampling",
    "03": "03_Pose_Training",
    "04": "04_Pose_Inference",
    "05": "05_Feature_Extraction",
    "06": "06_Feeding_Detection",
    "07": "07_Behavior_Classification",
    "08": "08_Reporting",
}

if NOTEBOOK_NUM not in NOTEBOOKS:
    print(f"Invalid number: {NOTEBOOK_NUM}. Choose from: {', '.join(NOTEBOOKS.keys())}")
else:
    nb_name = NOTEBOOKS[NOTEBOOK_NUM]
    nb_path = Path("notebooks") / f"{nb_name}.ipynb"

    # Pull latest code first (picks up opencode's fixes)
    !git pull

    print(f"\\n{'=' * 60}")
    print(f"Running: {nb_name}")
    print(f"{'=' * 60}\\n")

    # Build --path arguments for each Drive path variable
    path_args = []
    for var_name in DRIVE_PATH_NAMES:
        val = globals().get(var_name)
        if val:
            path_args.extend(["--path", f"{var_name}={val}"])

    result = subprocess.run(
        [sys.executable, "dev/colab_runner.py"] + path_args + [str(nb_path)],
        capture_output=True, text=True,
    )

    print(result.stdout)
    if result.stderr:
        print(result.stderr)

    if result.returncode == 0:
        print(f"\\n✔ {nb_name} PASSED. Move to the next notebook.")
    else:
        print(f"\\n✘ {nb_name} FAILED.")
        print(f"Run the next cell to see the error and instructions.")

In [ ]:
error_file = Path("/tmp/colab_error.txt")
if error_file.exists():
    print("=" * 60)
    print("ERROR TRACEBACK (saved to /tmp/colab_error.txt)")
    print("=" * 60)
    print(error_file.read_text())
    print("=" * 60)
    print("COPY EVERYTHING ABOVE and paste it to opencode.")
    print("After opencode fixes, re-run the previous cell.")
    print("It will git pull and re-execute.")
else:
    print("No error file found. The last notebook run was successful.")

---
## Quick Reference

| Action | Command |
|--------|---------|
| Run notebook 03 | Change `NOTEBOOK_NUM = "03"` in Step 3 cell, re-run |
| See last error | `cat /tmp/colab_error.txt` (or run the error display cell) |
| Pull latest fixes | `!git pull` (auto-done before each run) |
| Run a notebook directly | `!python dev/colab_runner.py --path DRIVE_ROOT=/my/path notebooks/03_Pose_Training.ipynb` |
| Run multiple in sequence | `!python dev/colab_runner.py --path DRIVE_ROOT=/my/path notebooks/01*.ipynb notebooks/02*.ipynb` |

**Where opencode runs:** In your local terminal (Mac) or wherever you
have this conversation open. opencode doesn't need to run on Colab —
just share the error and I'll fix it.